In [1]:
import os
from dotenv import load_dotenv
from ollama import chat

from LLM.Day02.first_cloud_chat import user_text, messages

In [2]:
def ask_local_model(question: str) -> str:
    load_dotenv()
    ollama_name = (os.getenv("OLLAMA_MODEL") or "").strip()
    if not ollama_name:
        raise RuntimeError('请配置 OLLAMA_MODEL环境')
    MAX_QUESTION_LENGTH = 200
    question = input('你: ' or '').strip()
    if not question:
        raise ValueError('question 不能为空')
    if len(question) > MAX_QUESTION_LENGTH:
        raise ValueError('question 文本长度不能超过200')
    messages = (
        {
            'role': 'system',
            'content': '你是一个ai助手，你的话少，内容简单易懂'
        },
        {
            'role': 'user',
            'content': question
        }
    )
    response = chat(
        model=ollama_name,
        messages=messages,
        stream=False
    )
    return response.message.content

In [3]:
TASK_PROMPTS = {
    "1": "请用通俗、简洁的语言解释输入内容，必要时给出一个例子；不确定时明确说明。",
    "2": "请提取输入内容的核心信息，整理为不超过三点的摘要；不要补充原文没有的事实。",
    "3": "请在不改变原意的前提下润色输入内容，使表达清晰自然；不要添加新的事实。",
}

def build_message(choice: str, user_text: str) -> list:
    choice = (choice or '').strip()
    if choice not in TASK_PROMPTS:
        raise ValueError('功能编号无效，请输入 1、2 或 3')

    user_text = (user_text or '').strip()
    if not user_text:
        raise ValueError('输入不能为空')
    return [
        {
            'role': 'system',
            'content': TASK_PROMPTS[choice]
        },
        {
            'role': 'user',
            'content': user_text
        }
    ]

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [5]:
MAX_INPUT_CHARS = 1000
load_dotenv()
api_key = (os.getenv("OPENAI_API_KEY") or "").strip()
model_name = (os.getenv("OPENAI_MODEL") or "").strip()
base_url = (os.getenv("BASE_URL") or "").strip()
if not api_key:
    raise RuntimeError('先设置 OPENAI_API_KEY')
if not model_name:
    raise RuntimeError('先设置 model_name')
if not base_url:
    raise RuntimeError('先设置 base_url')

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

In [13]:
messages = [
    {
        'role': 'system',
        'content': '你是需求澄清助手。信息不足时先追问，不自行编造条件。'
    }
]

In [18]:
while True:
    user_text = input('你: ').strip()
    if not user_text:
        print('请输入内容')
        continue
    if user_text.lower() in {'exit', 'quit'}:
        print('会话结束')
        break
    if len(user_text) > MAX_INPUT_CHARS:
        print(f'输入不能超过{MAX_INPUT_CHARS}个字符')
        continue
    messages.append(
        {
            'role': 'user',
            'content': user_text
        }
    )
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=messages
        )
        assistant_text = (
            response.choices[0].message.content
            or ''
        ).strip()
    except Exception as error:
        print(f'调用失败，本轮未保存: {type(error).__name__}')

    if not assistant_text:
        print('模型没有返回有效文本，本轮未保存')
        continue

    messages.append(
        {
            'role': 'assistant',
            'content': assistant_text
        }
    )
    print(f'助手: {assistant_text}')

KeyboardInterrupt: Interrupted by user

In [ ]:
usage = response.usage
if usage is not None:
    print(
        'Token 用量: ',
        f'输入={usage.prompt_tokens},'
        f'输出={usage.completion_tokens},'
        f'总计={usage.total_tokens}'
    )

In [19]:
SYSTEM_MESSAGE = {
    'role': 'system',
    'content': '你是需求澄清助手。'
}

In [20]:
def trim_history(message, max_rounds):
    if max_rounds < 0:
        raise ValueError('max_rounds 不能小于0')
    if not message or messages[0].get('role') != 'system':
        raise ValueError('消息历史必须从 system 开始')
    dialogue = message[1:]
    if len(dialogue) % 2 != 0:
        raise ValueError('消息历史不完整')
    for i in range(0, len(dialogue), 2):
        user_message = dialogue[i]
        assistant_message = dialogue[i + 1]
        if (
            user_message.get('role') != 'user'
            or assistant_message.get('role') != 'assistant'
        ):
            raise ValueError('消息历史必须按 user、assistant 顺序保存')

    keep_count = max_rounds * 2
    recent_dialogue = dialogue[-keep_count:] if keep_count else[]
    return [SYSTEM_MESSAGE.copy(), *recent_dialogue]

In [21]:
SYSTEM_MESSAGE = {
    "role": "system",
    "content": "你是需求澄清助手。信息不足时先追问，不自行编造条件。",
}
def strim_history(message, max_rounds):
    dialogue = message[1:]
    for i in range(0, len(dialogue), 2):
        user_message = dialogue[i]
        assistant_message = dialogue[i + 1]
    keep_count = max_rounds * 2
    recent_dialogue = dialogue[-keep_count:] if keep_count else []
    return [SYSTEM_MESSAGE.copy(), *recent_dialogue]

In [22]:
request_message = trim_history(messages, max_rounds=5)
request_message.append(
    {
        'role': 'user',
        'content': 'user_text'
    }
)
response = client.chat.completions.create(
    model=model_name,
    messages=request_message,
    stream=False
)
assistant_text = (response.choices[0].message.content or '').strip()
if not assistant_text:
    print('模型没有返回有效文本，本轮未保存')
messages.extent(
    [
        {
            'role': 'user',
            'content': user_text
        },
        {
            'role': 'assistant',
            'content': assistant_text
        }
    ]
)

ValueError: 消息历史不完整

In [25]:
from pathlib import Path

PROJECT_DIR = Path(__file__).resolve().parent
SESSION_PATH = PROJECT_DIR / 'session.json'


NameError: name '__file__' is not defined

In [ ]:
import json

def save_message(path, messages, max_saved_rounds):
    limited_messages = trim_history(messages, max_saved_rounds)
    dialogue = limited_messages[1:]
    path.write_text(
        json.dumps(dialogue, ensure_ascii=False, indent=4, sort_keys=True),
        encoding='utf-8'
    )

In [ ]:
def load_messages(path, max_saved_rounds):
    text = path.read_text(encoding='utf-8')
    data = json.loads(text)
    validated_dialogue = []
    for i, item in enumerate(data):
        excepted_role = 'user' if i % 2 == 0 else 'assistant'
        content = item.get('content')
        validated_dialogue.append(
            {
                'role': excepted_role,
                'content': content
            }
        )
    return [SYSTEM_MESSAGE.copy(), *validated_dialogue]

In [ ]:
user_text = input('你: ').strip()
stream = client.chat.completions.create(
    model=model_name,
    messages= [
        {
            'role': 'user',
            'content': user_text
        }
    ],
    stream=True,
    stream_options={'include_usage': True}
)
text_parts = []
finish_reason = None
usage = None
for chunk in stream:
    if chunk.usage is not None:
        usage = chunk.usage
    if not chunk.choices:
        continue
    choice = chunk.choices[0]
    delta_text = choice.delta.content or ''
    if delta_text:
        print(delta_text, end='', flush=True)
        text_parts.append(delta_text)
    if choice.finish_reason is not None:
        finish_reason = choice.finish_reason
print()
full_text = ''.join(text_parts).strip()

if finish_reason != 'stop' or not full_text:
    raise RuntimeError('流式响应未正常完成，不能写入成功历史')
print(f'长度: {len(full_text)}')
if usage is not None:
    print(
        usage.prompt_tokens,
        usage.completion_tokens,
        usage.total_tokens
    )

In [ ]:
import logging
import time
import uuid

logging.basicConfig(
    filename='llm_app.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    encoding='utf-8'
)

def log_result(trace_id, model_name, status, started_at, result):
    elapsed_ms = round((time.perf_counter() - started_at) * 1000)
    logging.info(
        "trace_id=%s model=%s status=%s elapsed_ms=%s "
        "output_chars=%s error_type=%s",
        trace_id,
        model_name,
        status,
        elapsed_ms,
        len(result.get("text", "")),
        result.get("error_type"),
    )
trace_id = uuid.uuid4().hex[:12]
started_at = time.perf_counter()

In [ ]:
def load_messages(path, max_saved_rounds):
    if not path.exists():
        return [SYSTEM_MESSAGE.copy()]
    text = path.read_text(encoding='utf-8')
    if not text.strip():
        return [SYSTEM_MESSAGE.copy()]
    try:
        data = json.load(text)
    except json.decoder.JSONDecodeError as error:
        raise ValueError(f"会话文件不是合法 JSON：{error}")
    if not isinstance(data, list):
        raise ValueError("会话文件顶层必须是列表")
    if len(data) % 2 != 0:
        raise ValueError("会话文件必须是完整问答组成的列表")
    if len(data) > max_saved_rounds * 2:
        raise ValueError(f"会话文件不能超过 {max_saved_rounds} 轮")
    validated_dialogue = []
    for i, item in enumerate(data):
        excepted_role = 'user' if i % 2 == 0 else 'assistant'
        if not isinstance(item, dict):
            raise ValueError
        content = item.get('content')
        if (
            item.get('role') != excepted_role
            or not isinstance(content, str)
            or not content.strip()
        ):
            raise ValueError
        validated_dialogue.append(
            {
                'role': excepted_role,
                'content': content
            }
        )
        return [SYSTEM_MESSAGE.copy(), *validated_dialogue]


In [ ]:
def load_messages(path, max_saved_rounds: int) -> list:
    if not path.exists():
        return [SYSTEM_MESSAGE.copy()]
    text = path.read_text(encoding='utf-8')
    if not text.strip():
        return [SYSTEM_MESSAGE.copy()]
    try:
        data = json.loads(text)
    except json.JSONDecodeError as error:
        raise ValueError(f'会话文件不合法: {error}')
    if not isinstance(data, list):
        raise ValueError('会话文件顶层必须是列表')
    if len(data) % 2 != 0:
        raise ValueError('会话文件必须完整')
    if len(data) > max_saved_rounds * 2:
        raise ValueError(f"会话文件不能超过 {max_saved_rounds} 轮")
    validated_dialogue = []
    for i, item in enumerate(data):
        expected_role = 'user' if i % 2 == 0 else 'assistant'
        if not isinstance(item, dict):
            raise ValueError('必须是字典')
        content = item.get('content')
        if (
            item.get('role') != expected_role
            or not isinstance(content, str)
            or not content.strip()
        ):
            raise ValueError(f"第 {i + 1} 条消息的角色或内容不正确")
        validated_dialogue.append(
            {
                'role': expected_role,
                'content': content
            }
        )
        return [SYSTEM_MESSAGE.copy(), *validated_dialogue]

In [ ]:
def collect_stream(stream) -> dict:
    text_report = []
    finish_reason = None
    usage = None
    for chunk in stream:
        if chunk.usage is not None:
            usage = chunk.usage
        if not chunk.choices:
            continue
        choice = chunk.choices[0]
        delta_text = choice.delta.content or ''
        if delta_text:
            print(delta_text, end='', flush=True)
            text_report.append(delta_text)
        if choice.finish_reason is not None:
            finish_reason = choice.finish_reason
    print()
    full_text = ''.join(text_report).strip()
    if finish_reason != 'stop' or not full_text:
        raise RuntimeError('流式响应未正常完成')
    print(f'完整长度: {len(full_text)}')
    if usage is not None:
        print(
            f'输入={usage.prompt_tokens},'
            f'输出={usage.completion_tokens},'
            f'总计={usage.total_tokens}'
        )
